# Answer key — Advanced session, tasks D-3 and D-4

The two tasks from `2026_2_Hands_on_session1_advanced.ipynb`, with the code filled in and the
outputs from a real run.

These cells continue from the main notebook — `df`, `candidates`, `find_candidates` and
`result_columns` are all defined there. To reproduce the outputs yourself, run the main notebook
from the top through B-3, then run the two cells below.


### D-3. Task 1 — change the chemistry

Everything so far started from sulfides. Do the same search for **oxides**: require `O`, exclude
`S`, and keep the 10⁻³ S/cm cutoff.

Call `find_candidates` yourself in the cell below and print three things.

1. How many confirmed candidates, and how many distinct formula strings.
2. The three highest conductivities, with their `record_id` and `formula`.
3. Whether the oxide list is longer or shorter than the sulfide list from B-3.

Everything you need is already defined: `find_candidates`, `df`, and the `result_columns` list.
Python set literals are written `{"O"}`.

In [15]:
## D-3 Task 1 — the same search with the chemistry swapped: oxides in, sulfides out.
oxide_candidates, oxide_uncertain = find_candidates(df, 1e-3, {"O"}, {"S"})

print("Confirmed candidates      :", len(oxide_candidates))
print("Distinct formula strings  :", oxide_candidates["formula"].nunique())
print("Uncertain upper-limit rows:", len(oxide_uncertain))
print()

## The sulfide numbers from B-3, so the two lists can be compared directly.
print("Sulfides from B-3         :", len(candidates), "records /",
      candidates["formula"].nunique(), "formulas")
print("Oxides are the shorter list under these conditions.")
print()

## find_candidates already sorts by conductivity, so the top three are the first three rows.
display(oxide_candidates[result_columns].head(3))


Confirmed candidates      : 27
Distinct formula strings  : 23
Uncertain upper-limit rows: 0

Sulfides from B-3         : 47 records / 42 formulas
Oxides are the shorter list under these conditions.



,record_id,formula,sigma_S_cm,family,doi
434,4p7,Li3ClO,0.025,antiperovskite,10.1039/c3ta15087a
285,mg4,Li1.3Al0.3Ti1.7(PO4)3,0.0062,NASICON,10.1016/j.ssi.2014.07.018
305,oxc,Li1.4Al0.4Ti1.6(PO4)3,0.00563,NASICON,10.1039/c5ta08545d


### D-4. Task 2 — the same material in more than one paper

In C-2 you compared records for a formula you picked by hand. Now let the table pick it for you.

Working from the `candidates` table produced in B-3:

1. Find the formula string that appears in the **most** records. `value_counts()` on the `formula`
   column sorts them for you, so the first entry of its index is the one you want.
2. Print those records with their `record_id`, `sigma_S_cm` and `doi`.
3. Work out the ratio between the highest and the lowest reported conductivity for that formula.
4. Write one sentence on what you would have to check in those papers before calling the difference
   real.

`table.loc[table["formula"].eq(name)]` selects the rows for one formula, the same way C-2 did.

In [16]:
## D-4 Task 2 — value_counts sorts by count, so the first index entry is the most repeated.
counts = candidates["formula"].value_counts()
repeated = counts.index[0]

## Select every candidate record carrying that formula, the way C-2 did.
rows = candidates.loc[candidates["formula"].eq(repeated)]

print("Formula appearing in the most records:", repeated)
print("Records:", len(rows), "| distinct DOIs:", rows["doi"].nunique())
print()
display(rows[["record_id", "sigma_S_cm", "sigma_total_S_cm", "sigma_bulk_S_cm", "doi"]])

## Report the spread as a ratio rather than a difference; conductivities span decades.
print("highest / lowest = %.1f" % (rows["sigma_S_cm"].max() / rows["sigma_S_cm"].min()))
print()

## What the table can and cannot settle. Check before assuming, rather than assuming.
print("total / bulk reported for these records:",
      int(rows[["sigma_total_S_cm", "sigma_bulk_S_cm"]].notna().sum().sum()))
print("total / bulk reported anywhere in the candidate list:",
      int(candidates[["sigma_total_S_cm", "sigma_bulk_S_cm"]].notna().sum().sum()))
print("records in the whole dataset reporting either:",
      int((df["sigma_total_S_cm"].notna() | df["sigma_bulk_S_cm"].notna()).sum()), "of", len(df))
print()

## Why the distinction matters, shown on a record that does report both.
example = df.loc[df["record_id"].eq("0o6"),
                 ["record_id", "formula", "sigma_total_S_cm", "sigma_bulk_S_cm"]]
display(example)
print("A total conductivity includes the grain boundaries and a bulk value does not,")
print("so for that sample the two numbers sit a factor of 25 apart.")
print()
print("Conclusion: the table cannot say whether these three laboratories measured the same")
print("quantity, because neither column is filled here. Only the three papers can, together")
print("with how each sample was synthesised and densified.")


Formula appearing in the most records: Li10SnP2S12
Records: 3 | distinct DOIs: 3



,record_id,sigma_S_cm,sigma_total_S_cm,sigma_bulk_S_cm,doi
191,9lx,0.007,NaN,NaN,10.1021/ja407393y
194,13q,0.00398,NaN,NaN,10.1021/acs.chemmater.8b00266
598,kko,0.0038,NaN,NaN,10.1021/jacs.0c10735


highest / lowest = 1.8

total / bulk reported for these records: 0
total / bulk reported anywhere in the candidate list: 0
records in the whole dataset reporting either: 30 of 599



,record_id,formula,sigma_total_S_cm,sigma_bulk_S_cm
232,0o6,Li1.2Ti1.8Sc0.2(PO4)3,9.98e-05,0.00251


A total conductivity includes the grain boundaries and a bulk value does not,
so for that sample the two numbers sit a factor of 25 apart.

Conclusion: the table cannot say whether these three laboratories measured the same
quantity, because neither column is filled here. Only the three papers can, together
with how each sample was synthesised and densified.
